In [11]:
import pandas as pd
import numpy as np

patients    = pd.read_csv('/home/suraj/Git/synthea/output/csv/patients.csv')
encounters  = pd.read_csv('/home/suraj/Git/synthea/output/csv/encounters.csv')
conditions  = pd.read_csv('/home/suraj/Git/synthea/output/csv/conditions.csv')
medications = pd.read_csv('/home/suraj/Git/synthea/output/csv/medications.csv')
procedures  = pd.read_csv('/home/suraj/Git/synthea/output/csv/procedures.csv')
observations= pd.read_csv('/home/suraj/Git/synthea/output/csv/observations.csv')



In [14]:
def build_event_stream(patients, conditions, medications, procedures, observations):
    events = []

    # Conditions → ICD codes (these map to PheCode in real pipeline)
    for _, row in conditions.iterrows():
        events.append({
            'patient_id': row['PATIENT'],
            'timestamp': pd.to_datetime(row['START']),
            'code': str(row['CODE']),
            'code_type': 'SNOMED',
            'description': row.get('DESCRIPTION', '')
        })

    # Medications → RxNorm (Synthea uses RxNorm natively — no mapping needed)
    for _, row in medications.iterrows():
        events.append({
            'patient_id': row['PATIENT'],
            'timestamp': pd.to_datetime(row['START'], utc=True),  # has time component
            'code': str(int(row['CODE'])),     # RxNorm integer → string
            'code_type': 'RxNorm',
            'description': row.get('DESCRIPTION', '')
        })

    # Procedures → CCS codes (simplified: use raw code for now)
    for _, row in procedures.iterrows():
        events.append({
            'patient_id': row['PATIENT'],
            'timestamp': pd.to_datetime(row['START']),
            'code': str(row['CODE']),
            'code_type': 'CCS'
        })

    # Labs/Observations
    for _, row in observations.iterrows():
        events.append({
            'patient_id': row['PATIENT'],
            'timestamp': pd.to_datetime(row['DATE']),
            'code': str(row['CODE']),
            'code_type': 'LAB'
        })

    df = pd.DataFrame(events).dropna(subset=['timestamp', 'code'])
    df = df.sort_values(['patient_id', 'timestamp'])
    return df

#all_events = build_event_stream(patients, conditions, medications, procedures, observations)

In [15]:
all_events = build_event_stream(patients, conditions, medications, procedures, observations)

TypeError: 'values' is not ordered, please explicitly specify the categories order by passing in a categories argument.

## Apply TALE-EHR Temporal Encoding

In [ ]:
def apply_tale_temporal_encoding(events_df):
    # Get each patient's first event time (t=0 anchor)
    first_event = events_df.groupby('patient_id')['timestamp'].min().rename('first_event')
    events_df = events_df.join(first_event, on='patient_id')

    # Compute relative time in weeks
    events_df['rel_time_weeks'] = (
        (events_df['timestamp'] - events_df['first_event'])
        .dt.total_seconds() / (7 * 24 * 3600)
    )

    # Log-transform (add 1 to avoid log(0))
    events_df['rel_time_log'] = np.log1p(events_df['rel_time_weeks'])

    return events_df.drop(columns=['first_event'])

all_events = apply_tale_temporal_encoding(all_events)

## Add Age as Time-Varying Covariate

In [ ]:
patients['BIRTHDATE'] = pd.to_datetime(patients['BIRTHDATE'])

def compute_age_at_event(events_df, patients_df):
    patient_births = patients_df.set_index('Id')['BIRTHDATE']
    events_df['birthdate'] = events_df['patient_id'].map(patient_births)
    events_df['age_at_event_years'] = (
        (events_df['timestamp'] - events_df['birthdate'])
        .dt.days / 365.25
    ).clip(0, 18)  # pediatric bound
    return events_df.drop(columns=['birthdate'])

all_events = compute_age_at_event(all_events, patients)

In [ ]:
import torch
import pickle

def build_tale_dataset(events_df, patients_df, max_seq_len=1024):
    dataset = {}
    patient_meta = patients_df.set_index('Id')

    for pid, group in events_df.groupby('patient_id'):
        group = group.sort_values('timestamp').head(max_seq_len)

        # Core sequence: list of (log_time, code, age_at_event)
        dataset[pid] = {
            'timestamps': torch.FloatTensor(group['rel_time_log'].values),      # (T,)
            'codes':      list(group['code'].values),                            # (T,) strings
            'ages':       torch.FloatTensor(group['age_at_event_years'].values), # (T,) — your age conditioning
            'sex':        patient_meta.loc[pid, 'GENDER'] if pid in patient_meta.index else 'U',
            'seq_len':    len(group)
        }

    return dataset

tale_dataset = build_tale_dataset(all_events, patients)

## Train/Test split

In [ ]:
from sklearn.model_selection import train_test_split

pids = list(tale_dataset.keys())
train_ids, temp_ids = train_test_split(pids, test_size=0.30, random_state=42)
val_ids,   test_ids = train_test_split(temp_ids, test_size=0.667, random_state=42)
# Result: 70% train, 10% val, 20% test

splits = {'train': train_ids, 'val': val_ids, 'test': test_ids}

## Save everything

In [ ]:
with open('tale_ehr_synthetic_pediatric.pkl', 'wb') as f:
    pickle.dump({'data': tale_dataset, 'splits': splits}, f)

print(f"Patients: {len(tale_dataset)}")
print(f"Train/Val/Test: {len(train_ids)}/{len(val_ids)}/{len(test_ids)}")

# Quick sanity check
sample = tale_dataset[train_ids[0]]
print(f"Seq len: {sample['seq_len']}")
print(f"Timestamps shape: {sample['timestamps'].shape}")
print(f"Age at first event: {sample['ages'][0]:.2f} years")
assert sample['ages'].max() <= 18, "Non-pediatric patient slipped through!"
print("✓ All checks passed")